### Прогоны тестов на добавление товаров со стороны модуля <font style="color:green;"><b><i>prestashop</i></b></font>

# <a id='toc1_'></a>[Amazon тест](#toc0_)

## `Murano Glass` - Прогоняю весь сценарий на добавление товаров со страницы категории


In [1]:
from pathlib import Path
from typing import Union
import notebook_header
from notebook_header import gs 
from notebook_header import logger,  logs_and_errors_decorator,  jprint, pprint
from notebook_header import SN, SF
from notebook_header import Product, ProductFields
from notebook_header import Supplier, start_supplier
from notebook_header import Driver
from src.scenarios import run_scenarios


s = start_supplier('amazon','en')
""" s - на протяжении всего кода означает класс `Supplier` """

print(" Можно продолжать ")





Master password for keepass database: ········


NoneType: None
2024-02-06 23:28:20,575 - ERROR - Error v2
NoneType: None
2024-02-06 23:28:20,607 - INFO - 
        -----------------------------------------
                Старт amazon
        -----------------------------------------
2024-02-06 23:28:20,630 - INFO - amazon Begin to attach driver ... 
2024-02-06 23:28:20,632 - INFO -  
        -----------------------------------------------------
        Старт вебдрайвера firefox. 
        Будет выбран драйвер, подключатся настройки и профиль.
        На моем компьютере это занимает около полуминуты. Ждем....
        -----------------------------------------------------
2024-02-06 23:28:20,633 - INFO -  Firefox ... 
2024-02-06 23:28:52,553 - INFO -  ... стартовал 
2024-02-06 23:28:52,568 - DEBUG -  ... драйвер подключился за 32 seconds 
NoneType: None


 Можно продолжать 



## Соглашения: 
-  <b><i>s</i></b> - поставщик  *(Supplier)*
-  <b><i>p</i></b> - товар  *(Product)*
-  <b><i>l</i></b> - локаторы полей товара *(locators)*
-  <b><i>d</i></b> - драйвер  *(Driver)*
-  <b><i>f</i></b> - поля товара  *(ProductFields)*
- <b><i>reread_locators</i></b> - перечитываль оригинальных локторов из файла

In [3]:

p: Product = Product(s)
l: dict = s.locators['product']
d: Driver = s.driver
f: ProductFields = ProductFields(s)
    

TypeError: 'mappingproxy' object does not support item assignment

In [ ]:
s.current_scenario: dict = {
      "url": "https://amzn.to/3OhRz2g",
      "condition": "new",
      "presta_categories": {
        "default_category": { "11209": "MURANO GLASS" },
        "addtinal categories": [ "" ]
      },
      "price_rule": 1
    }

s.run_scenario(s.current_scenario)

In [ ]:
ret = run_scenarios(s, s.current_scenario)

_______________________________________________
## <a id='toc1_3_'></a>[Поля товара](#toc0_)


`grab_product_page` <p>Это функция из модуля <font color="11073B"> suppliers.amazon.grabber</font>
Ее можно тестировать. 
<h6>Не забудь!</h6>
<font color='red'><b>Глявное - не забыть потом синхронизирoвать в suppliers.amazon.grabber</b><font></p>

### Проверяю наличие товара в бд

In [ ]:
product_reference = f"{s.supplier_id}-{_(l['reference'])}"
response = Product.check_if_product_in_presta_db(product_reference)
pprint(f' Если товар в бд получу id_product, иначе False. Получил: {response}')

In [ ]:


""" Я добавляю в базу данных престашоп товар путем нескольких последовательных действий
1. Заполняю поля, необходимые для создания нового товара
2. Получаю `id_product` созданного товара
3. Используя полученный `id_product` загружаю дефолтную картинку
4. итд.
"""

from typing import Union
# ----------------------------
from global_settings import gs
from src.helpers import  logger,  logs_and_errors_decorator,  jprint, pprint
from src.tools import SF, SN
from src.product import Product, ProductFields
from src.suppliers import Supplier
# ----------------------------




def grab_product_page(s: Supplier, id_product: int = 0 , api_method: Union[str('V1'),str('V2'),str('V3')] = 'V3') -> ProductFields:
    """ ПОКА ТОЛЬКО ДЛЯ НОВЫХ!!!!! СТРАНИЦ
        Собираю информацию со страницы товара. 
        Важно помнить, что драйвер уже должен быть на
        этой странице
        ---------------
        Attrs:
            s (Supplier)
        @returns
            f (ProductFields) с заполненными полями, else False
    """


    l = s.reread_locators('product')
    d = s.driver
   
    _ = d.execute_locator

    print('Strat local function grabber')

    def add_new_product_stage_1(s: Supplier) -> ProductFields:
        """ Первая стадия добавления нового товара. Заполняю все необходимые поля
            Далее я отправлю их в prestashop и получу обратно ID вновь созданного товара
            На второй стадии, зная ID, я отправлю главную картинку и прочее
        """
        f: ProductFields = ProductFields()
       
        """ ID,ASIN,SKU,SUPPLIER SKU """
    

        def _set_defaults() -> bool:
            """ Set defaults for product of supplier """
            f.active = 1
            f.on_sale = 1
            f.min_qty = 1
            f.low_stock_level = 0
            f.low_stock_threshold = ''
            f.show_price = 1
            f.show_condition = 1
            f.aviable_online_only = 0
            f.advanced_stock_management = 0
            f.state = 1

        def set_price(s, format: str = 'str') -> Union[str,float]:
            """ Привожу денюшку к 
            [ ] float 
            [v] str
            """
            l = s.reread_locators('product')
            raw_price = _(l['price'])[0]
            #raw_price = str(raw_price).split('\n')[0]
            return SN.normalize_price(raw_price)
    

        """ Я добавляю в базу данных престашоп товар путем нескольких последовательных действий
        1. Добавляю поля, необходимые для создания нового товара
        2. Получаю `id_product` созданного товара
        3. Используя полученный `id_product` загружаю дефолтную картинку
        4. итд.
        """

        _set_defaults()
        l = s.reread_locators('product')
        reference = _(l['reference'])
        f.reference = f'{s.supplier_id}-{reference}'
        f.supplier_reference = reference
        f.price = set_price(s, format = 'str')
        f.name = _(l['name'])[0]
        
        f.description_short = _(l['description_short'])[0]
        f.id_supplier = s.supplier_id
        
        _category_default = list(s.current_scenario['presta_categories']['default_category'].keys())[0]
        f.id_category_default = _category_default
        f.category_ids = Product.get_parent_categories(_category_default)

        #affiliate = _(l['affiliate short link'])
        #affiliate = affiliate[1][0]
        #f.affiliate_short_link = affiliate

        #f.link_rewrite = d.current_url.split(f'''/''')[-4]



        #f.link_rewrite = "test-link"
        return f

    def upload_image(id_product, image_url) -> ProductFields:
        """ После того, как я занес новый товар в бд - я получу его id
        Далее я гружу фото и получаю ее id
        Далее я догружаю осталные па
        ----------------------
        Attrs: 
            s (Supplier):
            f (ProductFields): Заполненные на первом этапе поля. Я беру из них только то, что мне надо для апдейта
        @returns
            ProductFields: поля для апдейта
        """

        img = Product.upload_product_default_image(id_product, image_url)
        return img
        
    



    if id_product == 0:
        print("---------------------NEW PRODUCT-----------------------")
        f = add_new_product_stage_1(s)
        product_dict = f.product_dict
        product = Product.add_2_prestashop(product_dict, api_method)
        id_product = product['id']
        print("---------------------NEW PRODUCT ID-----------------------")
        print(id_product)
        image_url = _(l['Image URLs (x,y,z...)'])[0]
        img = upload_image(id_product, image_url)
        return product, img

    
    pass


In [ ]:
product_fields: ProductFields = Product.grab_product_page(s)
#product_fields: ProductFields = grab_product_page(s)

In [ ]:
jprint(product_fields.fields)

In [ ]:
product_dict: dict = {}
product_dict['product'] = product_fields.fields

In [ ]:
jprint(product_dict)

In [ ]:
#prod = PrestaAPIV1.add('products', product_dict)

In [ ]:
pprint(prod['product'])

# Погоняй до сюда

In [ ]:
l = s.reread_locators('product')
affiliate = _(l['affiliate short link'])
pprint(l['affiliate short link'])
pprint(affiliate)

In [ ]:

product_dict['product']['link_rewrite']: dict = \
{
    "language":[
        {'attrs': {'id': '1'},'value': link_rewrite},
        {'attrs': {'id': '1'},'value': link_rewrite},
        {'attrs': {'id': '1'},'value': link_rewrite}
]}


product_dict['product']['affiliate_short_link']: dict = \
{
    "language":[
        {'attrs': {'id': '1'},'value': affiliate[1][0]},
        {'attrs': {'id': '1'},'value': affiliate[1][0]},
        {'attrs': {'id': '1'},'value': affiliate[1][0]}
]}



pprint(product_dict)

In [ ]:
prod = PrestaAPIV1OdAhad.add('products', product_dict)

In [ ]:
pprint(PrestaAPIV1OdAhad.add('products', product_dict))
#pprint(Product.add_2_prestashop(product_dict))

In [ ]:
PrestaAPIV1.add('products', product_dict)

In [ ]:
test_p = Product.get_product_info_by_id(63)